# TP2. Arithmétique et nombres premiers

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp2-arithmetique/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : la liste des diviseurs

Écrivez une fonction `diviseurs(n)` qui renvoie la liste des diviseurs positifs de `n`,
dans l'ordre croissant.

**Résultat attendu :** `diviseurs(28)` vaut `[1, 2, 4, 7, 14, 28]`.


In [1]:
def diviseurs(n):
    return [d for d in range(1, n + 1) if n % d == 0]

print(diviseurs(28))

[1, 2, 4, 7, 14, 28]


## Exercice 2.1 : les deux PGCD

Écrivez `pgcd_naif(a, b)`, qui essaie tous les entiers de 1 à `min(a, b)`, puis
`pgcd(a, b)`, qui applique Euclide. Vérifiez qu'ils donnent le même résultat.

**Résultat attendu :** `pgcd(1071, 462)` vaut `21`, comme `pgcd_naif(1071, 462)`.


In [2]:
def pgcd_naif(a, b):
    for d in range(min(a, b), 0, -1):
        if a % d == 0 and b % d == 0:
            return d

def pgcd(a, b):
    while b != 0:
        a, b = b, a % b
    return a

print(pgcd(1071, 462), pgcd_naif(1071, 462))

21 21


## Exercice 3.1 : le tableau des temps

Mesurez les deux fonctions sur les nombres premiers 10 007, 100 003 et 1 000 003, et
présentez les résultats. Que se passe-t-il quand `n` est multiplié par 10 ?

**Résultat attendu :** le temps de la version naïve est multiplié par 10 à chaque
ligne, celui de la version à racine carrée par 3 environ.


In [3]:
import time

def est_premier_naif(n):
    if n < 2:
        return False
    for d in range(2, n):
        if n % d == 0:
            return False
    return True

def est_premier(n):
    if n < 2:
        return False
    d = 2
    while d * d <= n:
        if n % d == 0:
            return False
        d += 1
    return True

def temps(f, n, fois=5):
    t = time.perf_counter()
    for _ in range(fois):
        f(n)
    return (time.perf_counter() - t) / fois

print("n         naif (s)   racine (s)")
for n in [10007, 100003, 1000003]:
    print(n, round(temps(est_premier_naif, n), 6), round(temps(est_premier, n), 6))

# Quand n est multiplié par 10 : le temps naïf est multiplié par 10 (il fait n tours),
# celui de la version racine carrée par environ 3 (il fait racine(n) tours, et racine(10) vaut 3,16).

n         naif (s)   racine (s)
10007 0.000358 7e-06
100003 0.003977 1.9e-05
1000003 0.040862 0.000105


## Exercice 4.1 : combien de premiers sous 100 000 ?

Comptez les nombres premiers inférieurs à 100 000, avec le crible, et mesurez le temps.
Comparez au temps qu'il faudrait en appelant `est_premier` sur chaque entier.

**Résultat attendu :** `9592` nombres premiers, obtenus en quelques centièmes de
seconde par le crible.


In [4]:
def crible(n):
    est = [True] * (n + 1)
    est[0] = est[1] = False
    for i in range(2, int(n ** 0.5) + 1):
        if est[i]:
            for j in range(i * i, n + 1, i):
                est[j] = False
    return [i for i in range(n + 1) if est[i]]

t = time.perf_counter()
print(len(crible(100000)))
print("crible :", round(time.perf_counter() - t, 4), "s")

t = time.perf_counter()
print(sum(1 for i in range(100000) if est_premier(i)))
print("est_premier sur chaque entier :", round(time.perf_counter() - t, 4), "s")

9592
crible : 0.0189 s
9592
est_premier sur chaque entier : 0.1863 s


## Exercice 5.1 : factoriser

Écrivez `facteurs(n)` qui renvoie la liste des facteurs premiers de `n`, avec leurs
répétitions.

**Résultat attendu :** `facteurs(360)` vaut `[2, 2, 2, 3, 3, 5]`, et le produit de
cette liste redonne `360`.


In [5]:
def facteurs(n):
    liste = []
    d = 2
    while d * d <= n:
        while n % d == 0:
            liste.append(d)
            n //= d
        d += 1
    if n > 1:
        liste.append(n)
    return liste

f = facteurs(360)
print(f)

p = 1
for x in f:
    p *= x
print(p)

[2, 2, 2, 3, 3, 5]
360


## Exercice 6.1 : mesurer la catastrophe

Mesurez le temps de `fibonacci_recursif` pour `n = 20`, `25` et `30`. Que devient le
temps quand `n` augmente de 5 ?

**Résultat attendu :** le temps est multiplié par une dizaine tous les cinq rangs, et
`fibonacci_recursif(30)` demande déjà un ou deux dixièmes de seconde.


In [6]:
def fibonacci_recursif(n):
    if n < 2:
        return n
    return fibonacci_recursif(n - 1) + fibonacci_recursif(n - 2)

for n in [20, 25, 30]:
    t = time.perf_counter()
    fibonacci_recursif(n)
    print(n, round(time.perf_counter() - t, 4), "s")

20 0.0008 s
25 0.0088 s
30 0.1004 s


## Exercice 6.2 : comprendre pourquoi

Ajoutez à `fibonacci_recursif` un compteur d'appels, par exemple un dictionnaire qui
note combien de fois chaque valeur de `n` est demandée. Combien de fois `F(1)` est-il
recalculé pour obtenir `F(30)` ?

**Résultat attendu :** `F(1)` est calculé `832040` fois, et l'appel total déclenche
environ 2,7 millions d'appels.


In [7]:
compteur = {}

def fibonacci_compte(n):
    compteur[n] = compteur.get(n, 0) + 1
    if n < 2:
        return n
    return fibonacci_compte(n - 1) + fibonacci_compte(n - 2)

fibonacci_compte(30)
print(compteur[1])
print(sum(compteur.values()))

832040
2692537


## Exercice 6.3 : la version qui se souvient

Écrivez `fibonacci(n)` avec une simple boucle qui fait avancer deux variables, la
valeur courante et la précédente. Vérifiez qu'elle donne les mêmes résultats que la
version récursive jusqu'à `n = 25`, puis calculez `fibonacci(10_000)`.

**Résultat attendu :** les deux versions coïncident ; `fibonacci(10_000)` a 2 090
chiffres et s'obtient en quelques millisecondes.


In [8]:
def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

print(all(fibonacci(n) == fibonacci_recursif(n) for n in range(26)))

t = time.perf_counter()
f = fibonacci(10_000)
print(len(str(f)), "chiffres")
print(round(time.perf_counter() - t, 4), "s")

True
2090 chiffres
0.0018 s


## Exercice 7.1 : l'algorithme d'Euclide étendu

L'algorithme d'Euclide donne le PGCD ; sa version étendue donne en plus deux entiers
$u$ et $v$ tels que $au + bv = \mathrm{pgcd}(a, b)$. Écrivez
`euclide_etendu(a, b)` qui renvoie le triplet `(pgcd, u, v)`. Indication : si
`(g, u, v)` convient pour `(b, a % b)`, alors `(g, v, u - (a // b) * v)` convient pour
`(a, b)`.

**Résultat attendu :** `euclide_etendu(1071, 462)` vaut `(21, -3, 7)`, et
$1071 \times (-3) + 462 \times 7 = 21$.


In [9]:
def euclide_etendu(a, b):
    if b == 0:
        return a, 1, 0
    g, u, v = euclide_etendu(b, a % b)
    return g, v, u - (a // b) * v

print(euclide_etendu(1071, 462))
print(1071 * (-3) + 462 * 7)

(21, -3, 7)
21


## Exercice 7.2 : l'exponentiation rapide

Calculer $a^{e} \bmod n$ en calculant d'abord $a^{e}$ est absurde quand
$e$ est grand : le nombre intermédiaire aurait des millions de chiffres. La
solution naïve fait `e` multiplications en réduisant modulo `n` à chaque tour. La
solution rapide remarque que $a^{2k} = (a^{k})^{2}$ : on élève au carré et on ne
multiplie par $a$ que lorsque le bit correspondant de $e$ vaut 1. Écrivez les
deux, `puissance_mod_naive(a, e, n)` et `puissance_mod(a, e, n)`, et comparez leurs
temps pour $7^{10^{7}} \bmod 1\,000\,003$.

**Résultat attendu :** le même résultat, `pow(a, e, n)` de Python le confirme, en une
seconde environ pour la version naïve et en une dizaine de microsecondes pour la
rapide.


In [10]:
def puissance_mod_naive(a, e, n):
    r = 1
    for _ in range(e):
        r = r * a % n
    return r

def puissance_mod(a, e, n):
    r = 1
    a = a % n
    while e > 0:
        if e % 2 == 1:
            r = r * a % n
        a = a * a % n
        e //= 2
    return r

t = time.perf_counter()
r1 = puissance_mod_naive(7, 10**7, 1000003)
t1 = time.perf_counter() - t

t = time.perf_counter()
r2 = puissance_mod(7, 10**7, 1000003)
t2 = time.perf_counter() - t

print(r1, r2, pow(7, 10**7, 1000003))
print("naive :", round(t1, 3), "s")
print("rapide :", round(t2 * 1e6), "microsecondes")

514944 514944 514944
naive : 0.497 s
rapide : 12 microsecondes
